# 🔐 MQTT Intrusion Detection System — MQTTset Dataset
**Phần 2: Machine Learning Pipeline cho MQTT IDS**

Pipeline:
1. Import & khám phá dataset
2. Tiền xử lý & feature engineering
3. Train mô hình (Random Forest, XGBoost, LSTM)
4. Đánh giá & so sánh kết quả
5. Lưu mô hình tốt nhất

## 📦 1. Cài đặt thư viện & Import Dataset

In [ ]:
# Cài đặt thư viện cần thiết
!pip install kagglehub xgboost imbalanced-learn seaborn -q

In [ ]:
import kagglehub
import os
import glob

# Download MQTTset dataset
path = kagglehub.dataset_download("cnrieiit/mqttset")
print("Path to dataset files:", path)

# Liệt kê toàn bộ file trong dataset
print("\n📂 Cấu trúc dataset:")
for root, dirs, files in os.walk(path):
    level = root.replace(path, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 2 * (level + 1)
    for file in files:
        filepath = os.path.join(root, file)
        size_mb = os.path.getsize(filepath) / (1024*1024)
        print(f'{subindent}{file}  ({size_mb:.1f} MB)')

## 📊 2. Load & Khám phá Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Tìm file CSV trong dataset
csv_files = glob.glob(os.path.join(path, '**/*.csv'), recursive=True)
print("📄 Các file CSV tìm thấy:")
for f in csv_files:
    print(f'  {os.path.basename(f)}')

In [ ]:
# Tìm file train và test
# Ưu tiên dùng bản augmented để xử lý class imbalance
def find_csv(keyword, prefer='augmented'):
    """Tìm file CSV theo keyword, ưu tiên bản augmented"""
    matches = [f for f in csv_files if keyword in f.lower()]
    preferred = [f for f in matches if prefer in f.lower()]
    if preferred:
        return preferred[0]
    return matches[0] if matches else None

# Tìm file train70 và test30
train_file = find_csv('train70', prefer='augmented')
test_file  = find_csv('test30')

# Fallback: nếu không tìm được theo pattern trên, dùng file đầu tiên
if train_file is None:
    print("⚠️ Không tìm thấy train70, dùng file CSV đầu tiên")
    train_file = csv_files[0]
if test_file is None:
    print("⚠️ Không tìm thấy test30, sẽ tự split từ train")

print(f"\n✅ Train file: {os.path.basename(train_file)}")
print(f"✅ Test file:  {os.path.basename(test_file) if test_file else 'Sẽ tự split'}")

In [ ]:
# Load dữ liệu
print("⏳ Đang load dữ liệu...")
df_train = pd.read_csv(train_file)

if test_file:
    df_test = pd.read_csv(test_file)
    print(f"Train shape: {df_train.shape}")
    print(f"Test shape:  {df_test.shape}")
else:
    from sklearn.model_selection import train_test_split
    df_train, df_test = train_test_split(df_train, test_size=0.3, random_state=42, stratify=df_train.iloc[:,-1])
    print(f"Train shape: {df_train.shape}")
    print(f"Test shape:  {df_test.shape}")

print("\n🔍 5 dòng đầu:")
df_train.head()

In [ ]:
# Khám phá tổng quan
print("=" * 60)
print("📊 TỔNG QUAN DATASET")
print("=" * 60)
print(f"Số features:   {df_train.shape[1] - 1}")
print(f"Số samples:    {df_train.shape[0]:,}")
print(f"Missing values: {df_train.isnull().sum().sum()}")

# Tìm cột label (thường là cột cuối hoặc tên 'label', 'target', 'class')
label_candidates = ['label', 'target', 'class', 'attack', 'type']
label_col = None
for col in df_train.columns:
    if col.lower() in label_candidates:
        label_col = col
        break
if label_col is None:
    label_col = df_train.columns[-1]  # dùng cột cuối

print(f"\n🏷️  Cột nhãn: '{label_col}'")
print("\n📈 Phân bố classes:")
class_counts = df_train[label_col].value_counts()
print(class_counts)
print(f"\nTỉ lệ (%):\n{(class_counts / len(df_train) * 100).round(2)}")

In [ ]:
# Visualize phân bố class
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Phân bố Traffic trong MQTTset', fontsize=14, fontweight='bold')

# Bar chart
colors = ['#2ecc71', '#e74c3c', '#e67e22', '#9b59b6', '#3498db']
class_counts.plot(kind='bar', ax=axes[0], color=colors[:len(class_counts)], edgecolor='black', linewidth=0.5)
axes[0].set_title('Số lượng mẫu mỗi loại traffic')
axes[0].set_xlabel('Loại traffic')
axes[0].set_ylabel('Số lượng')
axes[0].tick_params(axis='x', rotation=30)
for bar, count in zip(axes[0].patches, class_counts):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                f'{count:,}', ha='center', va='bottom', fontsize=9)

# Pie chart
axes[1].pie(class_counts.values, labels=class_counts.index,
            autopct='%1.1f%%', colors=colors[:len(class_counts)],
            startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
axes[1].set_title('Tỉ lệ phần trăm')

plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Đã lưu: class_distribution.png")

## 🔧 3. Tiền xử lý & Feature Engineering

In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif

# ---- Xác định features ----
feature_cols = [col for col in df_train.columns if col != label_col]

print(f"Tổng số features: {len(feature_cols)}")
print(f"\nKiểu dữ liệu:")
print(df_train[feature_cols].dtypes.value_counts())

# Phân loại features
numeric_cols   = df_train[feature_cols].select_dtypes(include=[np.number]).columns.tolist()
categoric_cols = df_train[feature_cols].select_dtypes(exclude=[np.number]).columns.tolist()

print(f"\nNumeric features:  {len(numeric_cols)}")
print(f"Categoric features: {len(categoric_cols)}")
if categoric_cols:
    print(f"  → {categoric_cols[:10]}")

In [ ]:
def preprocess(df, label_col, le_features=None, le_label=None, scaler=None, fit=True):
    """
    Tiền xử lý dataset:
    - Encode categorical features
    - Encode nhãn
    - Xử lý missing values
    - Scale numeric features
    """
    df = df.copy()
    
    # 1. Tách X và y
    X = df.drop(columns=[label_col])
    y = df[label_col]
    
    # 2. Encode categorical columns
    cat_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()
    if fit:
        le_features = {}
        for col in cat_cols:
            le = LabelEncoder()
            X[col] = le.fit_transform(X[col].astype(str))
            le_features[col] = le
    else:
        for col in cat_cols:
            if col in le_features:
                # Xử lý unseen labels
                known = set(le_features[col].classes_)
                X[col] = X[col].astype(str).apply(lambda v: v if v in known else le_features[col].classes_[0])
                X[col] = le_features[col].transform(X[col])
    
    # 3. Xử lý missing values
    X = X.fillna(X.median(numeric_only=True))
    X = X.fillna(0)
    
    # 4. Thay thế inf
    X = X.replace([np.inf, -np.inf], np.nan).fillna(0)
    
    # 5. Encode nhãn
    if fit:
        le_label = LabelEncoder()
        y_enc = le_label.fit_transform(y)
    else:
        y_enc = le_label.transform(y)
    
    # 6. Scale features
    if fit:
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
    else:
        X_scaled = scaler.transform(X)
    
    return X_scaled, y_enc, le_features, le_label, scaler, X.columns.tolist()

# Tiền xử lý
print("⏳ Tiền xử lý dữ liệu...")
X_train, y_train, le_feats, le_label, scaler, feature_names = preprocess(df_train, label_col, fit=True)
X_test, y_test, _, _, _, _ = preprocess(df_test, label_col, le_feats, le_label, scaler, fit=False)

print(f"✅ X_train: {X_train.shape}")
print(f"✅ X_test:  {X_test.shape}")
print(f"✅ Classes: {le_label.classes_}")
n_classes = len(le_label.classes_)

In [ ]:
# Feature correlation heatmap (top 20 features)
if X_train.shape[1] > 1:
    df_feat = pd.DataFrame(X_train, columns=feature_names)
    df_feat['label'] = y_train
    
    # Correlation với label
    corr_with_label = df_feat.corr()['label'].drop('label').abs().sort_values(ascending=False)
    top_features = corr_with_label.head(20).index.tolist()
    
    print("🔝 Top 10 features quan trọng nhất (correlation với label):")
    for i, (feat, corr) in enumerate(corr_with_label.head(10).items(), 1):
        print(f"  {i:2d}. {feat:<35} {corr:.4f}")
    
    # Heatmap
    fig, ax = plt.subplots(figsize=(12, 8))
    top_corr_matrix = df_feat[top_features + ['label']].corr()
    sns.heatmap(top_corr_matrix, annot=False, cmap='coolwarm', center=0,
                linewidths=0.5, ax=ax)
    ax.set_title('Correlation Matrix — Top 20 Features', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("✅ Đã lưu: correlation_heatmap.png")

## 🤖 4. Train Models

### 4.1 Random Forest (Baseline)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                              accuracy_score, f1_score, precision_score, recall_score)
import time

print("🌳 Training Random Forest...")
t0 = time.time()

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    min_samples_split=5,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
    verbose=0
)
rf_model.fit(X_train, y_train)
rf_time = time.time() - t0

# Dự đoán
y_pred_rf = rf_model.predict(X_test)

# Metrics
rf_acc = accuracy_score(y_test, y_pred_rf)
rf_f1  = f1_score(y_test, y_pred_rf, average='weighted')

print(f"\n✅ Random Forest kết quả:")
print(f"   Accuracy:        {rf_acc:.4f} ({rf_acc*100:.2f}%)")
print(f"   F1 (weighted):   {rf_f1:.4f}")
print(f"   Training time:   {rf_time:.1f}s")
print("\n📋 Classification Report:")
print(classification_report(y_test, y_pred_rf, target_names=le_label.classes_))

In [ ]:
# Feature Importance từ Random Forest
importances = rf_model.feature_importances_
feat_imp_df = pd.DataFrame({'feature': feature_names, 'importance': importances})
feat_imp_df = feat_imp_df.sort_values('importance', ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 7))
colors_imp = plt.cm.viridis(np.linspace(0.2, 0.9, len(feat_imp_df)))
bars = ax.barh(feat_imp_df['feature'][::-1], feat_imp_df['importance'][::-1],
               color=colors_imp, edgecolor='none')
ax.set_xlabel('Importance Score', fontsize=11)
ax.set_title('Top 20 Feature Importances — Random Forest', fontsize=13, fontweight='bold')
ax.grid(axis='x', alpha=0.3)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('feature_importance_rf.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Đã lưu: feature_importance_rf.png")

### 4.2 XGBoost

In [ ]:
from xgboost import XGBClassifier

print("⚡ Training XGBoost...")
t0 = time.time()

xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=8,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1,
    verbosity=0
)
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)
xgb_time = time.time() - t0

y_pred_xgb = xgb_model.predict(X_test)

xgb_acc = accuracy_score(y_test, y_pred_xgb)
xgb_f1  = f1_score(y_test, y_pred_xgb, average='weighted')

print(f"\n✅ XGBoost kết quả:")
print(f"   Accuracy:        {xgb_acc:.4f} ({xgb_acc*100:.2f}%)")
print(f"   F1 (weighted):   {xgb_f1:.4f}")
print(f"   Training time:   {xgb_time:.1f}s")
print("\n📋 Classification Report:")
print(classification_report(y_test, y_pred_xgb, target_names=le_label.classes_))

### 4.3 LSTM (Sequence-based)

In [ ]:
# Kiểm tra TensorFlow
try:
    import tensorflow as tf
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
    from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
    from tensorflow.keras.utils import to_categorical
    print(f"✅ TensorFlow {tf.__version__} sẵn sàng")
    TF_AVAILABLE = True
except ImportError:
    print("⚠️ TensorFlow chưa cài — bỏ qua LSTM")
    TF_AVAILABLE = False

In [ ]:
if TF_AVAILABLE:
    # Reshape cho LSTM: (samples, timesteps=1, features)
    X_train_lstm = X_train.reshape(X_train.shape[0], 1, X_train.shape[1])
    X_test_lstm  = X_test.reshape(X_test.shape[0], 1, X_test.shape[1])
    
    y_train_cat = to_categorical(y_train, num_classes=n_classes)
    y_test_cat  = to_categorical(y_test,  num_classes=n_classes)
    
    # Build model
    lstm_model = Sequential([
        LSTM(128, input_shape=(1, X_train.shape[1]), return_sequences=True),
        Dropout(0.3),
        LSTM(64, return_sequences=False),
        Dropout(0.3),
        BatchNormalization(),
        Dense(64, activation='relu'),
        Dropout(0.2),
        Dense(n_classes, activation='softmax')
    ])
    
    lstm_model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    lstm_model.summary()
    
    # Callbacks
    callbacks = [
        EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6)
    ]
    
    print("\n🔁 Training LSTM...")
    t0 = time.time()
    history = lstm_model.fit(
        X_train_lstm, y_train_cat,
        validation_data=(X_test_lstm, y_test_cat),
        epochs=30,
        batch_size=256,
        callbacks=callbacks,
        verbose=1
    )
    lstm_time = time.time() - t0
    
    y_pred_lstm = np.argmax(lstm_model.predict(X_test_lstm, verbose=0), axis=1)
    
    lstm_acc = accuracy_score(y_test, y_pred_lstm)
    lstm_f1  = f1_score(y_test, y_pred_lstm, average='weighted')
    
    print(f"\n✅ LSTM kết quả:")
    print(f"   Accuracy:        {lstm_acc:.4f} ({lstm_acc*100:.2f}%)")
    print(f"   F1 (weighted):   {lstm_f1:.4f}")
    print(f"   Training time:   {lstm_time:.1f}s")
    print("\n📋 Classification Report:")
    print(classification_report(y_test, y_pred_lstm, target_names=le_label.classes_))

    # Plot training history
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history.history['accuracy'],     label='Train', color='#3498db', linewidth=2)
    axes[0].plot(history.history['val_accuracy'], label='Val',   color='#e74c3c', linewidth=2)
    axes[0].set_title('LSTM Accuracy', fontweight='bold')
    axes[0].set_xlabel('Epoch')
    axes[0].legend()
    axes[0].grid(alpha=0.3)
    
    axes[1].plot(history.history['loss'],     label='Train', color='#3498db', linewidth=2)
    axes[1].plot(history.history['val_loss'], label='Val',   color='#e74c3c', linewidth=2)
    axes[1].set_title('LSTM Loss', fontweight='bold')
    axes[1].set_xlabel('Epoch')
    axes[1].legend()
    axes[1].grid(alpha=0.3)
    
    plt.suptitle('LSTM Training History', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('lstm_training_history.png', dpi=150, bbox_inches='tight')
    plt.show()

## 📈 5. So sánh & Đánh giá kết quả

In [ ]:
# Tổng hợp kết quả
results = {
    'Random Forest': {
        'accuracy': rf_acc,
        'f1': rf_f1,
        'precision': precision_score(y_test, y_pred_rf, average='weighted'),
        'recall':    recall_score(y_test, y_pred_rf, average='weighted'),
        'time': rf_time,
        'y_pred': y_pred_rf
    },
    'XGBoost': {
        'accuracy': xgb_acc,
        'f1': xgb_f1,
        'precision': precision_score(y_test, y_pred_xgb, average='weighted'),
        'recall':    recall_score(y_test, y_pred_xgb, average='weighted'),
        'time': xgb_time,
        'y_pred': y_pred_xgb
    }
}

if TF_AVAILABLE:
    results['LSTM'] = {
        'accuracy': lstm_acc,
        'f1': lstm_f1,
        'precision': precision_score(y_test, y_pred_lstm, average='weighted'),
        'recall':    recall_score(y_test, y_pred_lstm, average='weighted'),
        'time': lstm_time,
        'y_pred': y_pred_lstm
    }

# Bảng tổng hợp
print("\n" + "=" * 70)
print("📊 BẢNG SO SÁNH KẾT QUẢ")
print("=" * 70)
results_df = pd.DataFrame({
    name: {
        'Accuracy (%)':   f"{v['accuracy']*100:.2f}",
        'F1 Score':       f"{v['f1']:.4f}",
        'Precision':      f"{v['precision']:.4f}",
        'Recall':         f"{v['recall']:.4f}",
        'Train Time (s)': f"{v['time']:.1f}"
    }
    for name, v in results.items()
}).T
print(results_df.to_string())
print("=" * 70)

best_model_name = max(results.keys(), key=lambda k: results[k]['f1'])
print(f"\n🏆 Model tốt nhất: {best_model_name} (F1={results[best_model_name]['f1']:.4f})")

In [ ]:
# Biểu đồ so sánh
metrics = ['accuracy', 'f1', 'precision', 'recall']
metric_labels = ['Accuracy', 'F1 Score', 'Precision', 'Recall']
model_names = list(results.keys())
colors_models = ['#3498db', '#e74c3c', '#2ecc71']

fig, axes = plt.subplots(1, 4, figsize=(16, 5))
fig.suptitle('So sánh hiệu năng các mô hình — MQTTset IDS', fontsize=13, fontweight='bold')

for i, (metric, label) in enumerate(zip(metrics, metric_labels)):
    values = [results[m][metric] for m in model_names]
    bars = axes[i].bar(model_names, values,
                        color=colors_models[:len(model_names)],
                        edgecolor='black', linewidth=0.5, width=0.5)
    axes[i].set_ylim(0, 1.1)
    axes[i].set_title(label, fontweight='bold')
    axes[i].tick_params(axis='x', rotation=15)
    axes[i].grid(axis='y', alpha=0.3)
    axes[i].spines[['top', 'right']].set_visible(False)
    for bar, val in zip(bars, values):
        axes[i].text(bar.get_x() + bar.get_width()/2,
                      bar.get_height() + 0.01,
                      f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Đã lưu: model_comparison.png")

In [ ]:
# Confusion Matrix cho model tốt nhất
best_pred = results[best_model_name]['y_pred']
cm = confusion_matrix(y_test, best_pred)
cm_pct = cm.astype(float) / cm.sum(axis=1)[:, np.newaxis] * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(f'Confusion Matrix — {best_model_name}', fontsize=13, fontweight='bold')

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le_label.classes_, yticklabels=le_label.classes_,
            ax=axes[0], linewidths=0.5, cbar_kws={'shrink': 0.8})
axes[0].set_title('Số lượng')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
axes[0].tick_params(axis='x', rotation=30)

# Percentage
sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='YlOrRd',
            xticklabels=le_label.classes_, yticklabels=le_label.classes_,
            ax=axes[1], linewidths=0.5, cbar_kws={'shrink': 0.8, 'label': '%'})
axes[1].set_title('Tỉ lệ phần trăm (%)')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('confusion_matrix_best.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Đã lưu: confusion_matrix_best.png")

In [ ]:
# Per-class performance
from sklearn.metrics import precision_recall_fscore_support

p, r, f, s = precision_recall_fscore_support(y_test, best_pred, labels=range(n_classes))
per_class_df = pd.DataFrame({
    'Class': le_label.classes_,
    'Precision': p,
    'Recall': r,
    'F1-Score': f,
    'Support': s
})

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(n_classes)
width = 0.25

ax.bar(x - width, per_class_df['Precision'], width, label='Precision', color='#3498db', alpha=0.85)
ax.bar(x,         per_class_df['Recall'],    width, label='Recall',    color='#e74c3c', alpha=0.85)
ax.bar(x + width, per_class_df['F1-Score'],  width, label='F1-Score',  color='#2ecc71', alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(le_label.classes_, rotation=20, ha='right')
ax.set_ylim(0, 1.15)
ax.set_ylabel('Score')
ax.set_title(f'Per-class Performance — {best_model_name}', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('per_class_performance.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n📊 Per-class Performance:")
print(per_class_df.to_string(index=False, float_format='{:.4f}'.format))

## 💾 6. Lưu mô hình

In [ ]:
import joblib
import json

# Lưu model tốt nhất
if best_model_name == 'Random Forest':
    joblib.dump(rf_model, 'best_model_rf.pkl')
    print("✅ Đã lưu: best_model_rf.pkl")
elif best_model_name == 'XGBoost':
    joblib.dump(xgb_model, 'best_model_xgb.pkl')
    print("✅ Đã lưu: best_model_xgb.pkl")
elif best_model_name == 'LSTM' and TF_AVAILABLE:
    lstm_model.save('best_model_lstm.h5')
    print("✅ Đã lưu: best_model_lstm.h5")

# Lưu preprocessors
joblib.dump(scaler,   'scaler.pkl')
joblib.dump(le_label, 'label_encoder.pkl')
joblib.dump(le_feats, 'feature_encoders.pkl')
print("✅ Đã lưu: scaler.pkl, label_encoder.pkl, feature_encoders.pkl")

# Lưu metadata
metadata = {
    'best_model': best_model_name,
    'classes': le_label.classes_.tolist(),
    'n_features': X_train.shape[1],
    'feature_names': feature_names,
    'results': {
        name: {k: float(v) if k != 'y_pred' else None
               for k, v in vals.items()}
        for name, vals in results.items()
    }
}
with open('model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)
print("✅ Đã lưu: model_metadata.json")

print("\n" + "=" * 60)
print("🎉 HOÀN THÀNH! Tóm tắt:")
print("=" * 60)
print(f"✅ Best model:  {best_model_name}")
print(f"✅ Accuracy:    {results[best_model_name]['accuracy']*100:.2f}%")
print(f"✅ F1 Score:    {results[best_model_name]['f1']:.4f}")
print(f"✅ Classes:     {le_label.classes_.tolist()}")

## 🧪 7. Demo Inference (Real-time prediction)

In [ ]:
def predict_traffic(raw_sample_df, model, scaler, le_feats, le_label):
    """
    Dự đoán loại traffic cho 1 sample mới.
    raw_sample_df: DataFrame với đúng columns như lúc train (không có cột label)
    """
    sample = raw_sample_df.copy()
    
    # Encode categorical
    for col in sample.select_dtypes(exclude=[np.number]).columns:
        if col in le_feats:
            val = str(sample[col].iloc[0])
            known = set(le_feats[col].classes_)
            if val not in known:
                val = le_feats[col].classes_[0]
            sample[col] = le_feats[col].transform([val])
    
    # Fill missing & scale
    sample = sample.fillna(0).replace([np.inf, -np.inf], 0)
    sample_scaled = scaler.transform(sample)
    
    # Predict
    pred = model.predict(sample_scaled)[0]
    label = le_label.inverse_transform([pred])[0]
    
    # Probability nếu có
    proba = None
    if hasattr(model, 'predict_proba'):
        proba = model.predict_proba(sample_scaled)[0]
    
    return label, proba

# Test với 5 sample ngẫu nhiên từ test set
print("🧪 Demo inference với 5 mẫu ngẫu nhiên:")
print("-" * 50)

best_model_obj = rf_model if best_model_name == 'Random Forest' else xgb_model

sample_indices = np.random.choice(len(df_test), 5, replace=False)
for idx in sample_indices:
    row = df_test.iloc[[idx]].drop(columns=[label_col])
    true_label = df_test.iloc[idx][label_col]
    pred_label, proba = predict_traffic(row, best_model_obj, scaler, le_feats, le_label)
    status = "✅" if pred_label == true_label else "❌"
    print(f"  {status} True: {true_label:<20} Pred: {pred_label}")

print("\n✅ Pipeline hoàn chỉnh và sẵn sàng deploy!")